In [ ]:
import os
import openai
import json
import random
import time
import re
import pickle
import traceback
from transformers import LlamaTokenizer
from tqdm import tqdm
import pandas as pd
import warnings
from tqdm import tqdm
from openai import OpenAI
warnings.filterwarnings('ignore')

# GPT-4
openai.api_type = "azure"
openai.api_base = "https://YOUR-RESOURCE.openai.azure.com"
openai.api_version = "2023-07-01-preview"
openai.api_key = os.environ.get("AZURE_OPENAI_KEY", "")


In [ ]:
def get_completion(message,model):
    
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    response = client.chat.completions.create(
        model=model,
        messages=message,
        temperature=0.6,
    )

    # Using regex to find a 0 or 1 in the response
    match = re.search(r'\b(0|1)\b', response.choices[0].message.content)
    
    if match:
        return int(match.group(1))  # Returns the matched 0 or 1 as an integer
    else:
        print("Valid response not found, please check the model's output or adjust the prompt.")
        return None 
    # return response.choices[0].text
    # response.choices[0].message.content

def extract_answer(text):
    # Split the text using '### Answer:' as the delimiter
    parts = text.split('Answer:')
    
    # Return the part after the '### Answer:' marker, stripped of any leading or trailing whitespace
    return parts[1].strip() if len(parts) > 1 else ''

prompt = """You are a helpful and precise assistant for comparing the quality of the Assistant 1's answer and the Assistant 2's answer.
[The start of Question] 
{question}
[The end of Question]

[The Start of Assistant 1's Answer]
{assistant_answer1}
[The End of Assistant 1's Answer]

[The Start of Assistant 2's Answer]
{assistant_answer2}
[The End of Assistant 2's Answer]

We would like to request your feedback on the performance of the AI assistant in response to the user question.
"""


In [ ]:
print(len(alpaca),len(llama1),len(llama2),len(gpt4o),len(gpt3))

In [ ]:
import itertools
import numpy as np
# Assuming your existing functions and imports are here
# subs = ['anatomy','clinical_knowledge','medical_genetics']
subs = ['clinical_knowledge','medical_genetics', 'college_biology']

for sub in subs:

    with open(f"data/response/{sub}_alpaca_num=100.json", 'r') as f:
        alpaca = json.load(f)
        
    with open(f"data/response/{sub}_llama1_num=100.json", 'r') as f:
            llama1 = json.load(f)
    
    with open(f"data/response/{sub}_llama2_num=100.json", 'r') as f:
            llama2 = json.load(f)
    
    
    with open(f"data/response/{sub}_gpt4omini_num=100.json", 'r') as f:
            gpt4o = json.load(f)
    
    with open(f"data/response/{sub}_gpt3.5_num=100.json", 'r') as f:
           gpt3 = json.load(f)
    
    print(sub)    
    vnames = ['gpt3','alpaca', 'gpt4o', 'llama1',  'llama2']
    num = 100
    model = "gpt-4"
    # "gpt-4"
    out_data = []
    
    # Simulate data for demonstration purposes
    # globals().update({name: [{'response': 'Dummy response', 'prompt': f'Dummy prompt from {name}'} for _ in range(num)] for name in vnames})
    
    # Iterate over all unique pairs of variable namesf
    for variable_name1, variable_name2 in itertools.combinations(vnames, 2):
        # Bernoulli trial to decide whether to run the code for this pair
        if np.random.binomial(1, p=0.5):  # p is the probability of success (e.g., 0.5)
            data1 = globals()[variable_name1]
            data2 = globals()[variable_name2]
            print(f"\n============== Pair: {variable_name1} vs {variable_name2} ===============\n")
            
            out0 = np.zeros(num)
            for i in tqdm(range(num)):
                # print(f"\n============== Iteration {i}, Pair: {variable_name1} vs {variable_name2} ===============\n")
    
                # answer1 = extract_answer(data1[i]['response'])
                # answer2 = extract_answer(data2[i]['response'])
                question = data1[i]['prompt']
    
                prompt_str = (prompt.format(
                    question=question,
                    assistant_answer1=data1[i]['response'],
                    assistant_answer2=data2[i]['response'],
                    ) + 
                    # "\nPlease act like a judge and compare the assistant 1's answer to the assistant 2's answer. " +
                    # "Each answer receives an overall score on a scale of 0 to 10, where a higher score indicates better overall performance." + 
                    "Please first output a single line containing one value: 1 if the assistant 1's answer is better than the assistant 2's answer; 0 if not). The only output i want is a single value. " )
    
                message = [{"role": "system", "content": "You are a healthcare professional."},
                           {"role": "user", "content": prompt_str}]
                # message = prompt_str
                # print(prompt_str)
                gpt4 = get_completion(message, model)
                # print("\nGPT4:\n", gpt4, "\n")
                out0[i] = int(gpt4)
                # Append the output along with the variable names
    
            
            print(out0)
            # file_path = f"data/response/res/{sub}_{variable_name1}_{variable_name2}_num={num}.json"
            # # print(file_path)
            # # Saving the data to a JSON file
            # with open(file_path, 'w') as json_file:
            #     json.dump(out0, json_file, indent=4)
            
            out_data.append({
                # 'iteration': i,
                'variable_name1': variable_name1,
                'variable_name2': variable_name2,
                'GPT4_output': out0
            })
    
        
        # print(out_data)
    file_path = f"data/eval/{sub}_num={num}.json"
    print(file_path)
    for entry in out_data:
        if isinstance(entry['GPT4_output'], np.ndarray):
            entry['GPT4_output'] = entry['GPT4_output'].tolist()
    
    # Saving the data to a JSON file
    with open(file_path, 'w') as json_file:
        json.dump(out_data, json_file, indent=4)


In [ ]:
## add more pair
# Assuming your existing functions and imports are here
sub = 'medical_genetics'
# ['anatomy', 'clinical_knowledge', 'medical_genetics']
vnames = ['gpt3', 'alpaca', 'gpt4o', 'llama1', 'llama2']
num = 100
model = "gpt-4"

with open(f"data/response/{sub}_alpaca_num=100.json", 'r') as f:
    alpaca = json.load(f)
    
with open(f"data/response/{sub}_llama1_num=100.json", 'r') as f:
        llama1 = json.load(f)

with open(f"data/response/{sub}_llama2_num=100.json", 'r') as f:
        llama2 = json.load(f)


with open(f"data/response/{sub}_gpt4omini_num=100.json", 'r') as f:
        gpt4o = json.load(f)

with open(f"data/response/{sub}_gpt3.5_num=100.json", 'r') as f:
       gpt3 = json.load(f)


# Define predefined pairs
predefined_pairs = [
    ('llama2', 'gpt4o'),
    # ('gpt3', 'alpaca'),
    ('llama1', 'llama2')
]

# for sub in subs:
print(sub)
out_data = []

# Load existing data if available
file_path = f"data/eval/{sub}_num={num}.json"
try:
    with open(file_path, 'r') as json_file:
        out_data = json.load(json_file)
except FileNotFoundError:
    print(f"No existing data found for {sub}. Starting fresh.")

    # Create a set of existing pairs to avoid duplicates
    # existing_pairs = {(entry['variable_name1'], entry['variable_name2']) for entry in out_data}
    
    # Iterate over predefined pairs and random pairs
all_pairs = predefined_pairs
for variable_name1, variable_name2 in all_pairs:
    # Skip if this pair is already processed
    # if (variable_name1, variable_name2) in existing_pairs or (variable_name2, variable_name1) in existing_pairs:
        # continue

    # Bernoulli trial to decide whether to run the code for this pair (only for random pairs)
    # if (variable_name1, variable_name2) in predefined_pairs or np.random.binomial(1, p=0.5):  # p is the probability of success (e.g., 0.5)
    data1 = globals()[variable_name1]
    data2 = globals()[variable_name2]
    print(f"\n============== Pair: {variable_name1} vs {variable_name2} ===============\n")
    
    out0 = np.zeros(num)
    for i in tqdm(range(num)):
        question = data1[i]['prompt']

        prompt_str = (prompt.format(
            question=question,
            assistant_answer1=data1[i]['response'],
            assistant_answer2=data2[i]['response'],
            ) + 
            "Please first output a single line containing one value: 1 if the assistant 1's answer is better than the assistant 2's answer; 0 if not. The only output I want is a single value.")

        message = [{"role": "system", "content": "You are a healthcare professional."},
                   {"role": "user", "content": prompt_str}]
        
        gpt4 = get_completion(message, model)
        out0[i] = int(gpt4)

    print(out0)
    
    out_data.append({
        'variable_name1': variable_name1,
        'variable_name2': variable_name2,
        'GPT4_output': out0.tolist()  # Convert numpy array to list for JSON serialization
    })

print(file_path)

# Saving the combined data to a JSON file
with open(file_path, 'w') as json_file:
    json.dump(out_data, json_file, indent=4)
